# Sublime Symbols (Extraction and Minor Preprocessing Notebook)

**Based on the earlier EDA notebook, this notebook contains the code to do the following**:
 1) Extract Frames from videos via cv2 $\rightarrow$ kmeans method
 2) Generate image captions from those frames
 3) Output all these files

## Getting Directory Files (while ignoring items on ignore list)
- Ignore list is curated in section 1 of the workshop notebook

In [1]:
# pull ignore list
import pickle
with open('ignore_list_altright.pkl', 'rb') as file:
    ignore_list = pickle.load(file)

## Functions for Video Processing

### Primary Goal:
- **For the task of downstream video clustering to identify alt right symbolism/imagery**
  - Break our data into frames
  - Use those frames to create clip embedding representations of videos
  - Get representative frames for videos via k-means clustering of each video (use those for our video clustering pipeline)
  - Experiment with clustering techniques
-  **Saving Frames via K-Means clustering**
    - For this case, I use k=5 as a baseline (some videos are very short)
    - I want to try a case where I iterate through 5-20 as possible values for k, saving the k amount of frames based on that clusters silhouette score. Due to compute, I leave this case for later (can consider doing it via a random sample of 100 videos, and then seeing which one begets the best silhouette for us)
    - I've had to try many different optimizations, i've keyed them as different functions, but leave them here for your viewing
    - I settled upon: reducing image size, pca, MPS-->for my clustering framing task
 
- **This step also accomplishes our audio processing component**

In [6]:
import os
import gc
import glob
from pathlib import Path
from typing import List, Tuple, Optional
import cv2
import numpy as np
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.decomposition import PCA
import time
import tqdm

# Optional: PyTorch for MPS acceleration
try:
    import torch
    from torch import nn
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch not available. Install with: pip install torch")

In [17]:
def getVideoFilepaths(directories, extensions=None, excluded_paths=None):
    """
    recursively glob directories to get all video file paths.
    """
    if extensions is None:
        extensions = ['*.mp4', '*.avi', '*.mov', '*.mkv', '*.flv', '*.wmv', '*.m4v']
    
    if excluded_paths is None:
        excluded_paths = set()
    elif not isinstance(excluded_paths, set):
        excluded_paths = set(excluded_paths)
    
    video_files = []
    for directory in directories:
        for ext in extensions:
            pattern = os.path.join(directory, '**', ext)
            for filepath in glob.glob(pattern, recursive=True):
                if filepath not in excluded_paths:
                    video_files.append(filepath)
    
    return sorted(video_files)


def extractFramesFromVideo(video_path, max_frames=None, sample_rate=1, resize_to=(128, 128)):
    """
    extract frames from a video file with optional resizing for speed.
    """
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        return [], False
    
    frames = []
    frame_count = 0
    extracted_count = 0
    
    while True:
        ret, frame = cap.read()
        
        if not ret:
            break
        
        if frame_count % sample_rate == 0:
            if resize_to is not None:
                frame = cv2.resize(frame, resize_to, interpolation=cv2.INTER_AREA)
            
            frames.append(frame)
            extracted_count += 1
            
            if max_frames and extracted_count >= max_frames:
                break
        
        frame_count += 1
    
    cap.release()
    
    return frames, True


def clusterFramesKmeans(frames, k=5, method='minibatch', use_pca=True, pca_components=50):
    """
    k-means clustering with different strategies.
    """
    if len(frames) == 0:
        return np.array([]), []
    
    if len(frames) < k:
        k = len(frames)
    
    start_time = time.time()
    
    frame_shape = frames[0].shape
    flattened_frames = np.array([frame.flatten() for frame in frames])
    flattened_frames = flattened_frames.astype(np.float32) / 255.0
    
    if use_pca and flattened_frames.shape[1] > pca_components:
        pca = PCA(n_components=pca_components, random_state=42)
        flattened_frames = pca.fit_transform(flattened_frames)
    
    if method == 'pytorch_mps' and TORCH_AVAILABLE:
        representative_indices, labels = _clusterPytorchMps(flattened_frames, k)
    elif method == 'minibatch':
        representative_indices, labels = _clusterMinibatch(flattened_frames, k)
    elif method == 'mps_minibatch':
        representative_indices, labels = _clusterPytorchMpsMinibatch(flattened_frames, k)
    else:
        representative_indices, labels = _clusterStandard(flattened_frames, k)
    
    elapsed = time.time() - start_time
    
    return representative_indices, labels


def _clusterStandard(data, k):
    """
    standard k-means clustering.
    """
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(data)
    
    cluster_representatives = []
    for cluster_id in range(k):
        cluster_indices = np.where(labels == cluster_id)[0]
        if len(cluster_indices) == 0:
            continue
        centroid = kmeans.cluster_centers_[cluster_id]
        distances = np.linalg.norm(data[cluster_indices] - centroid, axis=1)
        closest_idx = cluster_indices[np.argmin(distances)]
        cluster_representatives.append(closest_idx)
    
    return np.array(cluster_representatives), labels.tolist()


def _clusterMinibatch(data, k):
    """
    minibatch k-means clustering.
    """
    kmeans = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=256, n_init=3)
    labels = kmeans.fit_predict(data)
    
    cluster_representatives = []
    centers = kmeans.cluster_centers_.copy()
    
    for cluster_id in range(k):
        cluster_indices = np.where(labels == cluster_id)[0]
        if len(cluster_indices) == 0:
            continue
        
        cluster_data = data[cluster_indices]
        centroid = centers[cluster_id]
        distances = np.linalg.norm(cluster_data - centroid, axis=1)
        closest_idx = cluster_indices[np.argmin(distances)]
        cluster_representatives.append(int(closest_idx))
        
        del cluster_indices, cluster_data, distances
    
    labels_list = [int(x) for x in labels]
    
    del kmeans, labels, centers
    gc.collect()
    
    return np.array(cluster_representatives), labels_list


def _clusterPytorchMps(data, k, max_iter=100):
    """
    pytorch k-means with mps acceleration.
    """
    if not TORCH_AVAILABLE:
        return _clusterMinibatch(data, k)
    
    if not torch.backends.mps.is_available():
        device = torch.device('cpu')
    else:
        device = torch.device('mps')
    
    X = torch.from_numpy(data).float().to(device)
    n_samples = X.shape[0]
    
    centroids = _kmeansPlusPlusInitTorch(X, k)
    
    for iteration in range(max_iter):
        distances = torch.cdist(X, centroids)
        labels = torch.argmin(distances, dim=1)
        
        new_centroids = torch.zeros_like(centroids)
        for i in range(k):
            mask = labels == i
            if mask.sum() > 0:
                new_centroids[i] = X[mask].mean(dim=0)
            else:
                new_centroids[i] = centroids[i]
        
        if torch.allclose(centroids, new_centroids, atol=1e-4):
            break
        
        centroids = new_centroids
    
    cluster_representatives = []
    labels_cpu = labels.cpu().numpy()
    
    for cluster_id in range(k):
        cluster_indices = np.where(labels_cpu == cluster_id)[0]
        if len(cluster_indices) == 0:
            continue
        
        cluster_mask = labels == cluster_id
        cluster_points = X[cluster_mask]
        centroid = centroids[cluster_id]
        
        distances = torch.norm(cluster_points - centroid, dim=1)
        local_min_idx = torch.argmin(distances).item()
        
        global_idx = cluster_indices[local_min_idx]
        cluster_representatives.append(global_idx)
    
    return np.array(cluster_representatives), labels_cpu.tolist()


def _clusterPytorchMpsMinibatch(data, k, batch_size=256, max_iter=100):
    """
    minibatch k-means with mps acceleration.
    """
    if not TORCH_AVAILABLE:
        return _clusterMinibatch(data, k)
    
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
    
    X = torch.from_numpy(data).float().to(device)
    n_samples = X.shape[0]
    
    centroids = _kmeansPlusPlusInitTorch(X, k)
    
    indices = torch.randperm(n_samples, device=device)
    
    for iteration in range(max_iter):
        for batch_start in range(0, n_samples, batch_size):
            batch_end = min(batch_start + batch_size, n_samples)
            batch_indices = indices[batch_start:batch_end]
            batch = X[batch_indices]
            
            distances = torch.cdist(batch, centroids)
            labels = torch.argmin(distances, dim=1)
            
            for i in range(k):
                mask = labels == i
                if mask.sum() > 0:
                    learning_rate = 1.0 / (iteration + 1)
                    centroids[i] = (1 - learning_rate) * centroids[i] + \
                                   learning_rate * batch[mask].mean(dim=0)
        
        indices = torch.randperm(n_samples, device=device)
    
    distances = torch.cdist(X, centroids)
    labels = torch.argmin(distances, dim=1)
    
    cluster_representatives = []
    labels_cpu = labels.cpu().numpy()
    
    for cluster_id in range(k):
        cluster_indices = np.where(labels_cpu == cluster_id)[0]
        if len(cluster_indices) == 0:
            continue
        
        cluster_mask = labels == cluster_id
        cluster_points = X[cluster_mask]
        centroid = centroids[cluster_id]
        
        distances = torch.norm(cluster_points - centroid, dim=1)
        local_min_idx = torch.argmin(distances).item()
        global_idx = cluster_indices[local_min_idx]
        cluster_representatives.append(global_idx)
    
    return np.array(cluster_representatives), labels_cpu.tolist()


def _kmeansPlusPlusInitTorch(X, k):
    """
    k-means++ initialization in pytorch.
    """
    n_samples = X.shape[0]
    centroids = []
    
    first_idx = torch.randint(0, n_samples, (1,)).item()
    centroids.append(X[first_idx])
    
    for _ in range(1, k):
        centroids_tensor = torch.stack(centroids)
        distances = torch.cdist(X, centroids_tensor).min(dim=1)[0]
        probs = distances / distances.sum()
        next_idx = torch.multinomial(probs, 1).item()
        centroids.append(X[next_idx])
    
    return torch.stack(centroids)


def saveRepresentativeFrames(frames, representative_indices, video_path,
                              output_dir='representative_frames', original_frames=None):
    """
    save representative frames with traceable naming.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    video_name = Path(video_path).stem
    video_parent = Path(video_path).parent.name
    
    saved_paths = []
    frames_to_save = original_frames if original_frames is not None else frames
    
    for i, frame_idx in enumerate(representative_indices):
        if frame_idx >= len(frames_to_save):
            continue
        
        output_filename = f"{video_parent}_{video_name}_cluster{i:02d}_frame{frame_idx:05d}.jpg"
        output_path = os.path.join(output_dir, output_filename)
        
        cv2.imwrite(output_path, frames_to_save[frame_idx])
        saved_paths.append(output_path)
    
    return saved_paths


def processVideo(video_path, k=5, output_dir='representative_frames', max_frames=None,
                 sample_rate=1, clustering_method='minibatch', resize_for_clustering=(224, 224),
                 use_pca=True, pca_components=50, save_original_resolution=True):
    """
    process video pipelining function.
    """
    frames, success = extractFramesFromVideo(
        video_path, max_frames, sample_rate, resize_to=resize_for_clustering
    )
    
    if not success or len(frames) == 0:
        return []
    
    original_frames = None
    if save_original_resolution:
        original_frames, _ = extractFramesFromVideo(
            video_path, max_frames, sample_rate, resize_to=None
        )
    
    representative_indices, labels = clusterFramesKmeans(
        frames, k, method=clustering_method, use_pca=use_pca, pca_components=pca_components
    )
    
    saved_paths = saveRepresentativeFrames(
        frames, representative_indices, video_path, output_dir, original_frames
    )
    
    return saved_paths


def processDirectories(directories, k=5, output_dir='representative_frames', max_frames=None,
                       sample_rate=1, clustering_method='minibatch',
                       resize_for_clustering=(224, 224), use_pca=True, filter_list=None):
    """
    process all videos with optimized settings.
    """
    video_files = getVideoFilepaths(directories, excluded_paths=filter_list)
    print(f"found {len(video_files)} video files")
    print(f"settings: method={clustering_method}, resize={resize_for_clustering}, pca={use_pca}")
    
    results = {}
    total_start = time.time()

    for i, video_path in tqdm.tqdm(enumerate(video_files, 1), total=len(video_files),
                                    desc="processing videos", unit="video"):
        try:
            saved_paths = processVideo(
                video_path, k, output_dir, max_frames, sample_rate,
                clustering_method, resize_for_clustering, use_pca
            )
            results[video_path] = saved_paths
        except Exception as e:
            tqdm.tqdm.write(f"error processing {video_path}: {str(e)}")
            results[video_path] = []

        if i % 10 == 0:
            gc.collect()
    
    total_time = time.time() - total_start
    print(f"all videos processed in {total_time:.2f} seconds ({total_time/60:.1f} minutes)")
    
    return results

In [18]:
# processing alt videos
directories_to_process = ["videos/alt_right_2"]

results = processDirectories(
    directories=directories_to_process, k=5, output_dir="videos/representative_frames/alt",
    max_frames=None, sample_rate=1, clustering_method='minibatch',
    resize_for_clustering=(128, 128), use_pca=True, filter_list=ignore_list)

Found 942 video files
Optimization settings: method=minibatch, resize=(128, 128), PCA=True



Processing videos: 100%|█████████████████| 942/942 [2:16:37<00:00,  8.70s/video]

ALL VIDEOS PROCESSED in 8197.68 seconds (136.6 minutes)


# Image Captioning Step

In [1]:
import torch
import tqdm
import gc
from pathlib import Path
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM
import glob

In [2]:
def generateImageCaptions(image_dir, output_dir, model_name="microsoft/Florence-2-large",
                          image_extensions=("*.jpg", "*.jpeg", "*.png", "*.webp", "*.bmp"),
                          device="mps"):
    """
    generate detailed captions for images using florence-2.
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    image_paths = []
    for ext in image_extensions:
        image_paths.extend(glob.glob(f"{image_dir}/{ext}"))
        image_paths.extend(glob.glob(f"{image_dir}/{ext.upper()}"))
    
    print(f"found {len(image_paths)} images to process")
    
    if device == "mps" and not torch.backends.mps.is_available():
        device = "cpu"
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True
    ).to(device)
    
    processor = AutoProcessor.from_pretrained(
        model_name,
        trust_remote_code=True
    )
    
    for idx, image_path in enumerate(tqdm.tqdm(image_paths, desc="processing images"), 1):
        image = Image.open(image_path).convert("RGB")
        
        prompt = "<MORE_DETAILED_CAPTION>"
        
        inputs = processor(
            text=prompt,
            images=image,
            return_tensors="pt"
        )
        
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            generated_ids = model.generate(
                input_ids=inputs["input_ids"],
                pixel_values=inputs["pixel_values"],
                max_new_tokens=1024,
                num_beams=3,
                do_sample=False
            )
        
        generated_text = processor.batch_decode(
            generated_ids,
            skip_special_tokens=False
        )[0]
        
        parsed_answer = processor.post_process_generation(
            generated_text,
            task=prompt,
            image_size=(image.width, image.height)
        )

        caption = parsed_answer.get("<MORE_DETAILED_CAPTION>", "")
        
        image_filename = Path(image_path).stem
        caption_filename = f"CAPTION_{image_filename}.txt"
        caption_path = Path(output_dir) / caption_filename
     
        with open(caption_path, 'w', encoding='utf-8') as f:
            f.write(caption)
        
        del image, inputs, generated_ids, generated_text, parsed_answer
        
        if idx % 10 == 0:
            gc.collect()
            if device == "mps":
                torch.mps.empty_cache()
            elif device == "cuda":
                torch.cuda.empty_cache()
    
    del model, processor
    gc.collect()
    if device == "mps":
        torch.mps.empty_cache()
    
    print(f"all captions saved to: {output_dir}")
    return 0

In [3]:
MODEL = "microsoft/Florence-2-base"

generateImageCaptions(
    image_dir="videos/representative_frames/alt",
    output_dir="videos/representative_frames/captions",
    model_name=MODEL)

Found 4710 images to process


Processing images: 100%|██████████████████| 4710/4710 [2:27:15<00:00,  1.88s/it]

All captions saved to: videos/representative_frames/captions


0